# DL Masterclass 07: The Transformer Architecture from Scratch

**Track:** Deep Learning Theory & Practice  
**Prerequisites:** DL 01 (Neural Network Mathematics), DL 02 (Activations & Gradients)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–150 minutes

---

## 🎓 Socratic Deep Check

> *"The Attention mechanism computes a weighted sum over ALL tokens. This means it has O(n²) memory complexity for sequence length n. How does a model like GPT-4 handle 128K token contexts without exploding GPU memory?"*

---

## Why Transformers Are the Entire Industry

Every major AI model since 2017 is a Transformer:
- **GPT-4, Claude, Gemini** — decoder-only transformers
- **BERT** — encoder-only transformer
- **T5, BART** — encoder-decoder transformers
- **Vision Transformers (ViT)** — transformers for images
- **Whisper** — transformers for audio

If you understand this notebook, you understand the architecture behind every model in the curriculum.

## Table of Contents
1. Self-Attention from First Principles
2. Multi-Head Attention
3. Positional Encoding
4. The Full Transformer Block
5. Encoder vs Decoder Architectures

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors treat transformers as black boxes. Seniors understand that attention IS a differentiable key-value database lookup, and that design choices like KV-cache (NB18), Flash Attention, and Grouped-Query Attention are all optimizations of the same Q·K^T·V operation.

**Mental Model:** Imagine you're at a party. Self-attention is each person looking at everyone else and deciding who to pay attention to based on relevance to their current thought (Query). Each person broadcasts what they know (Value) and a label for what they know about (Key).

**Common Junior Pitfall:** Confusing the attention WEIGHTS (the soft routing pattern) with the attention VALUES (the actual information being routed). The attention matrix tells you WHERE to look; the values tell you WHAT you see there.

---

In [ ]:
# Cell 1 — Install
!pip install -q torch numpy matplotlib

## 1 · Self-Attention from First Principles

### The Problem Attention Solves

In the sentence: *"The cat sat on the mat because **it** was tired"*

What does "it" refer to? The cat, the mat, or the sitting? A human instantly knows it's the cat. Attention gives neural networks this same ability — each token can "look at" every other token to understand context.

### The Three Projections: Query, Key, Value

For each token, we compute three vectors:

| Vector | Analogy | Purpose |
|--------|---------|----------|
| **Query (Q)** | "What am I looking for?" | The question this token asks |
| **Key (K)** | "What do I contain?" | The label advertising this token's info |
| **Value (V)** | "Here's my actual content" | The information to retrieve |

### The Attention Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

| Step | Operation | Shape | What It Does |
|------|-----------|-------|--------------|
| 1 | $QK^T$ | (seq, seq) | Score every token pair's relevance |
| 2 | $/ \sqrt{d_k}$ | (seq, seq) | Prevent scores from getting too large |
| 3 | softmax | (seq, seq) | Normalize to probabilities (rows sum to 1) |
| 4 | $\times V$ | (seq, d_v) | Weighted combination of values |

In [ ]:
# Cell 2 — Self-Attention from scratch
import torch
import torch.nn.functional as F
import numpy as np

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    THE core operation of all transformers.
    
    Args:
        Q: Queries (batch, seq_len, d_k)
        K: Keys    (batch, seq_len, d_k)
        V: Values  (batch, seq_len, d_v)
        mask: Optional causal mask (for decoder/GPT-style models)
    
    Returns:
        output: Weighted values (batch, seq_len, d_v)
        weights: Attention pattern (batch, seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute attention scores
    # Q @ K^T = how relevant is each key to each query?
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, seq, seq)
    
    # Step 2: Scale (prevents softmax saturation for large d_k)
    scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    
    # Step 3: Apply causal mask (for decoder models like GPT)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 4: Softmax (convert to probabilities)
    weights = F.softmax(scores, dim=-1)
    
    # Step 5: Weighted sum of values
    output = torch.matmul(weights, V)  # (batch, seq, d_v)
    
    return output, weights

# Demo: "The cat sat" — 3 tokens, embedding dim = 4
torch.manual_seed(42)
seq_len, d_model = 3, 4
tokens = ['The', 'cat', 'sat']

# Simulated embeddings
X = torch.randn(1, seq_len, d_model)

# Project to Q, K, V (in real transformers, these are nn.Linear layers)
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

Q = X @ W_q
K = X @ W_k
V = X @ W_v

output, weights = scaled_dot_product_attention(Q, K, V)

print('Attention Weights (who pays attention to whom):')
print(f'         {"  ".join(f"{t:>6}" for t in tokens)}')
for i, token in enumerate(tokens):
    row = weights[0, i].detach().numpy()
    print(f'  {token:>4}: {"  ".join(f"{w:>6.3f}" for w in row)}')
print(f'\nEach row sums to 1.0: {weights[0].sum(dim=-1).tolist()}')

---
## 2 · Multi-Head Attention

### Why Multiple Heads?

A single attention head captures ONE type of relationship. Multiple heads capture DIFFERENT relationships simultaneously:

| Head | What It Might Learn |
|------|--------------------|
| Head 1 | Syntactic (subject-verb agreement) |
| Head 2 | Semantic (word meaning similarity) |
| Head 3 | Positional (nearby tokens) |
| Head 4 | Coreference ("it" → "cat") |

```
Input → [Head 1] → pattern 1 ─┐
      → [Head 2] → pattern 2 ─┼→ Concatenate → Linear → Output
      → [Head 3] → pattern 3 ─┘
```

In [ ]:
# Cell 3 — Multi-Head Attention from scratch
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention — the core of every transformer.
    
    This is EXACTLY what runs inside GPT, BERT, Llama, etc.
    The q_proj, k_proj, v_proj you see in NB11's LoRA target_modules are these projections.
    """
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Each head gets a slice
        
        # These are the q_proj, k_proj, v_proj from NB11's LoRA config!
        self.W_q = nn.Linear(d_model, d_model)  # Query projection
        self.W_k = nn.Linear(d_model, d_model)  # Key projection
        self.W_v = nn.Linear(d_model, d_model)  # Value projection
        self.W_o = nn.Linear(d_model, d_model)  # Output projection
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # 1. Project to Q, K, V
        Q = self.W_q(x)  # (batch, seq, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 2. Reshape to multiple heads: (batch, seq, d_model) -> (batch, heads, seq, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Scaled dot-product attention (per head)
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # 4. Concatenate heads: (batch, heads, seq, d_k) -> (batch, seq, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # 5. Final linear projection
        return self.W_o(attn_output)

# Demo
d_model, num_heads, seq_len = 64, 4, 10
mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(2, seq_len, d_model)  # Batch of 2 sequences
output = mha(x)

print(f'Multi-Head Attention')
print(f'  d_model={d_model}, heads={num_heads}, d_k={d_model//num_heads}')
print(f'  Input:  {x.shape}  (batch=2, seq=10, d_model=64)')
print(f'  Output: {output.shape}  (same shape — attention doesn\'t change dimensions)')
print(f'  Parameters: {sum(p.numel() for p in mha.parameters()):,}')
print(f'\n  Connection to NB11: When LoRA targets ["q_proj", "k_proj", "v_proj"],')
print(f'  it adds adapters to W_q, W_k, W_v — these exact projections.')

---
## 3 · Positional Encoding

### The Problem

Attention treats input as a SET (order doesn't matter). But "dog bites man" ≠ "man bites dog". We need to inject position information.

### Sinusoidal Positional Encoding (Original Transformer)

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Each position gets a unique sinusoidal fingerprint. The model learns to decode relative distances from these patterns.

### Modern Alternatives

| Method | Used By | Max Length | Learned? |
|--------|---------|-----------|----------|
| Sinusoidal | Original Transformer | Any | No |
| Learned | BERT, GPT-2 | Fixed (512-2048) | Yes |
| **RoPE** | **Llama, Mistral, Phi** | Extensible | No |
| ALiBi | MPT, BLOOM | Extensible | No |

In [ ]:
# Cell 4 — Positional Encoding implementation
import torch
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_positional_encoding(max_len, d_model):
    """Original transformer positional encoding."""
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)  # Even dimensions
    pe[:, 1::2] = torch.cos(position * div_term)  # Odd dimensions
    return pe

pe = sinusoidal_positional_encoding(100, 64)

plt.figure(figsize=(12, 4))
plt.imshow(pe.numpy().T, aspect='auto', cmap='RdBu')
plt.xlabel('Token Position')
plt.ylabel('Encoding Dimension')
plt.title('Sinusoidal Positional Encoding — Each Position Has a Unique Pattern')
plt.colorbar()
plt.tight_layout()
plt.savefig('positional_encoding.png', dpi=100)
plt.show()

print('Low dimensions (bottom): change fast — encode LOCAL position')
print('High dimensions (top):   change slow — encode GLOBAL position')
print(f'\nPosition 0 encoding (first 8 dims): {pe[0, :8].tolist()}')
print(f'Position 1 encoding (first 8 dims): {pe[1, :8].tolist()}')

---
## 4 · The Complete Transformer Block

A transformer is a STACK of identical blocks. Each block has:

```
Input
  │
  ├──→ [Multi-Head Attention] ──→ Add & LayerNorm ──→ output1
  │         (self-attention)          (residual connection)
  │
  └──→ [Feed-Forward Network] ──→ Add & LayerNorm ──→ output2
           (2 linear layers)         (residual connection)
```

- **Residual connections** (the `Add`) prevent vanishing gradients in deep networks
- **LayerNorm** stabilizes training (normalizes each sample independently)
- **FFN** adds non-linear processing capacity

In [ ]:
# Cell 5 — Complete Transformer Block
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    """One transformer block — GPT stacks 32-96 of these."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Multi-Head Attention
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        
        # Feed-Forward Network (expand then compress)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),    # Expand: d_model -> 4*d_model
            nn.GELU(),                    # Activation (smoother than ReLU)
            nn.Linear(d_ff, d_model),    # Compress back: 4*d_model -> d_model
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Attention + Residual + Norm
        attn_out = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_out))  # Residual connection
        
        # FFN + Residual + Norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))   # Residual connection
        
        return x

class MiniGPT(nn.Module):
    """A tiny GPT-style decoder-only transformer.
    
    Real GPT-4: d_model=12288, heads=96, layers=~120, params=1.8T
    Our MiniGPT: d_model=64, heads=4, layers=4, params=~100K
    """
    
    def __init__(self, vocab_size=1000, d_model=64, num_heads=4, num_layers=4, max_len=128):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff=d_model*4)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)  # Predict next token
    
    def forward(self, token_ids):
        B, T = token_ids.shape
        tok_emb = self.token_embedding(token_ids)
        pos_emb = self.position_embedding(torch.arange(T, device=token_ids.device))
        x = tok_emb + pos_emb
        
        # Causal mask: each token can only attend to previous tokens
        mask = torch.tril(torch.ones(T, T, device=token_ids.device)).unsqueeze(0)
        
        for block in self.blocks:
            x = block(x, mask)
        
        x = self.ln_final(x)
        logits = self.head(x)  # (batch, seq, vocab_size)
        return logits

# Create and inspect
model = MiniGPT(vocab_size=1000, d_model=64, num_heads=4, num_layers=4)
total_params = sum(p.numel() for p in model.parameters())

print(f'MiniGPT Architecture')
print(f'  Parameters: {total_params:,}')
print(f'  Layers: 4 transformer blocks')
print(f'  d_model: 64, heads: 4, d_k: 16')

# Forward pass
tokens = torch.randint(0, 1000, (2, 20))  # Batch of 2, seq_len=20
logits = model(tokens)
print(f'\n  Input:  {tokens.shape} (batch=2, seq=20, token IDs)')
print(f'  Output: {logits.shape} (batch=2, seq=20, vocab=1000)')
print(f'  Each position predicts a probability distribution over 1000 tokens')

# Compare to real models
print(f'\n📊 Scale Comparison:')
for name, params, layers, d in [('MiniGPT (ours)', total_params, 4, 64),
                                  ('GPT-2 Small', 124e6, 12, 768),
                                  ('Llama 3 8B', 8e9, 32, 4096),
                                  ('GPT-4 (est.)', 1.8e12, 120, 12288)]:
    print(f'  {name:>20}: {params:>13,.0f} params, {layers:>3} layers, d={d}')

---
## 5 · Encoder vs Decoder Architectures

| Architecture | Attention Type | Models | Best For |
|-------------|---------------|--------|---------|
| **Encoder-only** | Bidirectional (see all tokens) | BERT, RoBERTa | Classification, NER, embeddings |
| **Decoder-only** | Causal (see only past tokens) | GPT, Llama, Mistral | Text generation, chat |
| **Encoder-Decoder** | Both | T5, BART, Whisper | Translation, summarization |

### Causal Masking (Why GPT Can't See the Future)

```
Encoder (BERT):         Decoder (GPT):
  The  cat  sat           The  cat  sat
   ↕    ↕    ↕             ↓    ↓    ↓
  All see everything      Each sees only past

Attention mask:           Attention mask:
  [1, 1, 1]               [1, 0, 0]   The → sees only The
  [1, 1, 1]               [1, 1, 0]   cat → sees The, cat
  [1, 1, 1]               [1, 1, 1]   sat → sees The, cat, sat
```

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How do models handle 128K token contexts without O(n²) memory explosion?*

**A:** Multiple techniques (covered in NB18 Inference Optimization):
1. **Flash Attention** — reorders computation to minimize GPU memory reads (2-4x speedup)
2. **KV-Cache** — stores computed K,V from past tokens so they're not recomputed
3. **Grouped-Query Attention (GQA)** — shares K,V heads (Llama 3 uses this)
4. **Sliding Window Attention** — only attend to last N tokens (Mistral)
5. **Ring Attention** — distribute across multiple GPUs

---

## ✅ Knowledge Check

### Q1: Why scale by √d_k?
In the attention formula, why divide by √d_k instead of d_k or nothing?

<details><summary>👉 Click to reveal answer</summary>

When d_k is large (e.g., 128), the dot products Q·K^T produce very large numbers. Large inputs to softmax push outputs toward 0 or 1 (saturation), creating near-zero gradients. Dividing by √d_k keeps the variance of the dot products at ~1.0 regardless of dimension, preventing softmax saturation and maintaining healthy gradients.
</details>

### Q2: Residual Connections
Why does `x = norm(x + attn(x))` work better than `x = norm(attn(x))`?

<details><summary>👉 Click to reveal answer</summary>

The residual connection (`x + attn(x)`) creates a "gradient highway" — during backpropagation, gradients can flow directly through the addition operation without going through the attention layers. This prevents vanishing gradients in deep networks (96+ layers). Without residuals, training a 32-layer transformer would be nearly impossible.
</details>

### Q3: LoRA Connection
In NB11, LoRA targets `["q_proj", "k_proj", "v_proj", "dense"]`. What are these in terms of what we built?

<details><summary>👉 Click to reveal answer</summary>

`q_proj` = our `W_q` (Query projection), `k_proj` = our `W_k` (Key projection), `v_proj` = our `W_v` (Value projection). These are the `nn.Linear` layers inside Multi-Head Attention. `dense` refers to the feed-forward layers in the FFN block. LoRA adds small adapter matrices to these projections instead of modifying the original weights.
</details>

### Q4: Encoder vs Decoder
Why can't you use a decoder-only model (GPT) for computing sentence embeddings like BERT does?

<details><summary>👉 Click to reveal answer</summary>

A decoder uses causal masking — each token only sees tokens BEFORE it. The embedding for token 1 has zero information about tokens 2, 3, etc. BERT's encoder uses bidirectional attention — every token sees ALL other tokens — so the [CLS] token's embedding genuinely represents the entire sentence's meaning. For embeddings, you need bidirectional attention.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Attention Visualization
Modify Cell 2 to create a 6-token sentence and visualize the attention weights as a heatmap using `plt.imshow()`. Which token attends most to which?

### Exercise 2: Parameter Counting
Calculate the exact number of parameters in one TransformerBlock with d_model=512, num_heads=8, d_ff=2048. Break it down by component (MHA projections, FFN, LayerNorms).

### Exercise 3: Causal Masking
Create a causal mask for seq_len=5 and verify that applying it to attention scores makes future positions have weight 0 after softmax.

---

## Summary

| Concept | What You Learned | Curriculum Connection |
|---------|-----------------|----------------------|
| Self-Attention | Q·K^T·V weighted lookup | Core of every LLM |
| Multi-Head Attention | Parallel attention patterns | NB11 LoRA target_modules |
| Positional Encoding | Inject sequence order | RoPE in Llama (NB18) |
| Transformer Block | Attention + FFN + Residuals | NB11 model architecture |
| Encoder vs Decoder | Bidirectional vs causal | BERT vs GPT (NB27, 28) |